# Weak Signals Ensemble — The Medallion Approach

**Philosophy:** Don't optimize individual indicators. Instead, combine many *mediocre* signals with textbook parameters. The edge comes from **diversification across uncorrelated weak forecasts**, not from any single strong signal.

> *"The best approach is to use many different rules, each with equal weight... The performance of the equal-weight portfolio is almost always better than picking the best rule."* — Robert Carver, *Systematic Trading*

> *"We don't override the models."* — Jim Simons

**How this differs from the original ensemble notebook:**

| | Original (tema_macdv_rsi) | This Notebook |
|---|---|---|
| Indicators | 3 (TEMA, MACD-V, RSI) | 8 weak signals |
| Parameters | Grid-search optimized per indicator | Textbook defaults only |
| Signal type | Binary (on/off) | Continuous forecast (-1 to +1) |
| Combination | Voting (AND/OR/MAJORITY) | Equal-weighted average score |
| Edge source | Optimized parameters | Signal diversity (low correlation) |
| Overfit risk | High (many degrees of freedom) | Low (zero optimization on indicators) |

**Signal Menu (8 weak forecasts):**
1. **TEMA** — Triple EMA alignment (trend direction)
2. **MACD-V** — Volatility-normalized momentum
3. **RSI** — Mean-reversion timing
4. **ADX** — Trend strength filter
5. **Stochastic** — Overbought/oversold oscillator
6. **Bollinger %B** — Price position relative to volatility bands
7. **OBV Trend** — Volume confirmation
8. **Donchian** — Channel breakout (pure price)

---

## 1. Environment & Dependencies

In [ ]:
# !pip install yfinance TA-Lib vectorbt scipy pandas numpy matplotlib --quiet

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt
from itertools import product

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 140)

print("All imports loaded")
print(f"   vectorbt: {vbt.__version__}  |  pandas: {pd.__version__}  |  numpy: {np.__version__}")
from lib.data_manager import download, download_multi, setup_colab_secrets

## 2. Configuration — Textbook Defaults Only

### No grid search. No optimization. Every parameter below is a standard textbook value.
### The ONLY thing we tune later is the ensemble entry threshold (and we test that on training data).

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MASTER CONFIGURATION — TEXTBOOK DEFAULTS, ZERO OPTIMIZATION
# ═══════════════════════════════════════════════════════════════

TICKER         = "BTC-USD"        # Asset under test
START_DATE     = "2018-01-01"     # Download start
END_DATE       = None             # None = latest available
TRAIN_RATIO    = 0.60             # 60% train, 40% OOS
ANNUAL_FACTOR  = 365              # 365 for crypto, 252 for equities

# ─── TEXTBOOK INDICATOR PARAMETERS (NO OPTIMIZATION) ───
# TEMA: Classic triple EMA periods
TEMA_FAST      = 10
TEMA_MED       = 30
TEMA_SLOW      = 70

# MACD-V: Standard 12/26/9, normalized by ATR
MACDV_FAST     = 12
MACDV_SLOW     = 26
MACDV_SIGNAL   = 9
ATR_LEN        = 14

# RSI: Wilder's original
RSI_LEN        = 14
RSI_OVERSOLD   = 30
RSI_OVERBOUGHT = 70

# ADX: Wilder's original
ADX_LEN        = 14
ADX_TREND_THRESH = 25    # >25 = trending

# Stochastic: Standard fast stochastic
STOCH_K        = 14
STOCH_D        = 3
STOCH_OVERSOLD = 20
STOCH_OVERBOUGHT = 80

# Bollinger Bands: Standard 20/2
BB_LEN         = 20
BB_STD         = 2.0

# OBV: 20-period EMA of OBV for trend
OBV_EMA_LEN    = 20

# Donchian: Turtle-style 20-period channel
DONCH_LEN      = 20

# ─── Regime Filter ───
SMA_REGIME_LEN = 200

# ─── Ensemble Settings ───
# Entry when average forecast > threshold, exit when < -threshold
ENTRY_THRESHOLDS = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35]  # Test on training data
EXIT_THRESHOLD   = 0.0    # Exit when composite score goes negative
MIN_TRADES       = 10

# ─── Execution ───
SIGNAL_SHIFT   = 1              # Anti-lookahead bar shift
FEES           = 0.001          # 0.1% per trade (one-way)
SLIPPAGE       = 0.0005         # 0.05% slippage
INIT_CASH      = 100_000

# ─── FTMO Rules ───
FTMO_ACCOUNT       = 100_000
FTMO_PROFIT_TARGET = 0.10       # 10%
FTMO_MAX_DAILY_DD  = 0.05       # 5%
FTMO_MAX_TOTAL_DD  = 0.10       # 10%
FTMO_CHALLENGE_DAYS = 30
MONTE_CARLO_PATHS  = 10_000

print("Configuration loaded — ZERO optimization, textbook defaults only")
print(f"   Asset: {TICKER} | Dates: {START_DATE} -> {END_DATE or 'today'}")
print(f"   Split: {TRAIN_RATIO:.0%} train / {1-TRAIN_RATIO:.0%} OOS")
print(f"   8 weak signals: TEMA, MACD-V, RSI, ADX, Stochastic, Bollinger, OBV, Donchian")
print(f"   Entry thresholds to test: {ENTRY_THRESHOLDS}")
print(f"   Signal shift: {SIGNAL_SHIFT} bar(s) anti-lookahead")

## 3. Data Download & Validation

In [ ]:
# Data: yfinance (daily) | Alpaca (intraday) — auto-selected by data_manager
# For Colab: run setup_colab_secrets() if using Alpaca intraday features
raw_df = download(TICKER, START_DATE)

df = raw_df.copy()

required = ['Open', 'High', 'Low', 'Close', 'Volume']
missing = [c for c in required if c not in df.columns]
assert len(missing) == 0, f"Missing columns: {missing}"

# Validate
assert df.index.is_monotonic_increasing, "Index NOT time-ordered!"
assert df.index.is_unique, "Duplicate dates found!"

nan_pre = df.isna().sum().sum()
df = df.ffill().bfill()

close  = df['Close'].astype(float)
high   = df['High'].astype(float)
low    = df['Low'].astype(float)
open_  = df['Open'].astype(float)
volume = df['Volume'].astype(float)

print(f"\n{TICKER}: {len(df)} rows | {df.index[0].date()} -> {df.index[-1].date()}")
print(f"   NaNs filled: {nan_pre} | Shape: {df.shape}")
print(f"   Price range: {close.min():.2f} -> {close.max():.2f}")

## 4. Shared Components — ATR & Regime Filter

In [ ]:
# ATR — used by MACD-V and for position sizing reference
atr = pd.Series(talib.ATR(high.values, low.values, close.values, timeperiod=ATR_LEN), index=close.index)

# Regime filter (200-SMA)
sma_200 = pd.Series(talib.SMA(close.values, timeperiod=SMA_REGIME_LEN), index=close.index)
regime_bull = close > sma_200

print(f"ATR({ATR_LEN}): mean={atr.mean():.2f}, current={atr.iloc[-1]:.2f}")
print(f"Regime SMA-{SMA_REGIME_LEN}: {regime_bull.sum()} bullish bars ({regime_bull.mean():.1%})")
print(f"   Current: {'BULL' if regime_bull.iloc[-1] else 'BEAR'}")

## 5. The 8 Weak Signal Generators — Continuous Forecasts

Each function returns a **continuous forecast** from -1 (max bearish) to +1 (max bullish).

This is the key difference from binary signals. A forecast of +0.3 means "slightly bullish" — alone it's useless, but averaged across 8 uncorrelated forecasts, the signal emerges from the noise.

**Why continuous?** Binary signals throw away information. RSI at 31 vs RSI at 69 are both "neutral" in a binary system, but they carry very different information. Continuous forecasts preserve the gradient.

In [ ]:
def forecast_tema(close_s, fast=TEMA_FAST, med=TEMA_MED, slow=TEMA_SLOW):
    """TEMA: Trend direction from triple EMA alignment.
    +1 = perfect bull alignment (fast>med>slow), -1 = perfect bear alignment.
    Intermediate values based on how many pairs are aligned."""
    ema_f = pd.Series(talib.EMA(close_s.values, timeperiod=fast), index=close_s.index)
    ema_m = pd.Series(talib.EMA(close_s.values, timeperiod=med), index=close_s.index)
    ema_s = pd.Series(talib.EMA(close_s.values, timeperiod=slow), index=close_s.index)
    
    # Score: +1 for each bullish pair, -1 for each bearish pair, normalized to [-1, +1]
    score = ((ema_f > ema_m).astype(float) - (ema_f < ema_m).astype(float) +
             (ema_m > ema_s).astype(float) - (ema_m < ema_s).astype(float) +
             (ema_f > ema_s).astype(float) - (ema_f < ema_s).astype(float)) / 3.0
    return score


def forecast_macdv(close_s, atr_s, fast=MACDV_FAST, slow=MACDV_SLOW, signal=MACDV_SIGNAL):
    """MACD-V: Volatility-normalized momentum.
    Normalized MACD histogram mapped to [-1, +1] via clipping."""
    macd_line = pd.Series(
        talib.EMA(close_s.values, timeperiod=fast) - talib.EMA(close_s.values, timeperiod=slow),
        index=close_s.index
    )
    macd_v = macd_line / atr_s.replace(0, np.nan)
    sig_line = pd.Series(talib.EMA(macd_v.values, timeperiod=signal), index=close_s.index)
    histogram = macd_v - sig_line
    
    # Normalize to [-1, +1] using rolling z-score, then clip
    hist_mean = histogram.rolling(100, min_periods=20).mean()
    hist_std = histogram.rolling(100, min_periods=20).std().replace(0, np.nan)
    z = (histogram - hist_mean) / hist_std
    return z.clip(-2, 2) / 2.0  # Map [-2, 2] -> [-1, 1]


def forecast_rsi(close_s, length=RSI_LEN, oversold=RSI_OVERSOLD, overbought=RSI_OVERBOUGHT):
    """RSI: Mean-reversion signal.
    Maps RSI to [-1, +1]: oversold = +1 (buy), overbought = -1 (sell).
    Linear interpolation between thresholds."""
    rsi = pd.Series(talib.RSI(close_s.values, timeperiod=length), index=close_s.index)
    midpoint = (oversold + overbought) / 2.0
    half_range = (overbought - oversold) / 2.0
    # Invert: low RSI = bullish (mean reversion), high RSI = bearish
    score = -(rsi - midpoint) / half_range
    return score.clip(-1, 1)


def forecast_adx(high_s, low_s, close_s, length=ADX_LEN, threshold=ADX_TREND_THRESH):
    """ADX: Trend strength with directional bias.
    Uses +DI vs -DI for direction, ADX magnitude for conviction.
    Strong uptrend = +1, strong downtrend = -1, no trend = 0."""
    adx = pd.Series(talib.ADX(high_s.values, low_s.values, close_s.values, timeperiod=length), index=close_s.index)
    plus_di = pd.Series(talib.PLUS_DI(high_s.values, low_s.values, close_s.values, timeperiod=length), index=close_s.index)
    minus_di = pd.Series(talib.MINUS_DI(high_s.values, low_s.values, close_s.values, timeperiod=length), index=close_s.index)
    
    # Direction: +DI vs -DI
    direction = (plus_di - minus_di) / (plus_di + minus_di).replace(0, np.nan)
    # Strength: ADX normalized (0-100 -> 0-1), only counts if above threshold
    strength = (adx / 100.0).clip(0, 1)
    strength = strength * (adx > threshold).astype(float)  # Zero out weak trends
    
    return (direction * strength).clip(-1, 1)


def forecast_stochastic(high_s, low_s, close_s, k=STOCH_K, d=STOCH_D,
                         oversold=STOCH_OVERSOLD, overbought=STOCH_OVERBOUGHT):
    """Stochastic: Overbought/oversold oscillator.
    Maps %K to [-1, +1] similar to RSI (inverted for mean reversion)."""
    slowk, slowd = talib.STOCH(high_s.values, low_s.values, close_s.values,
                                fastk_period=k, slowk_period=d, slowd_period=d)
    slowk = pd.Series(slowk, index=close_s.index)
    
    midpoint = (oversold + overbought) / 2.0
    half_range = (overbought - oversold) / 2.0
    score = -(slowk - midpoint) / half_range
    return score.clip(-1, 1)


def forecast_bollinger(close_s, length=BB_LEN, std_dev=BB_STD):
    """Bollinger %B: Position within bands.
    %B > 1 = above upper band (bearish mean-reversion), %B < 0 = below lower (bullish).
    Mapped to [-1, +1]."""
    upper, middle, lower = talib.BBANDS(close_s.values, timeperiod=length, nbdevup=std_dev, nbdevdn=std_dev)
    upper = pd.Series(upper, index=close_s.index)
    lower = pd.Series(lower, index=close_s.index)
    
    band_width = (upper - lower).replace(0, np.nan)
    pct_b = (close_s - lower) / band_width  # 0 to 1 normally
    
    # Invert for mean reversion: below lower band = bullish, above upper = bearish
    score = -(pct_b - 0.5) * 2.0
    return score.clip(-1, 1)


def forecast_obv(close_s, volume_s, ema_len=OBV_EMA_LEN):
    """OBV Trend: Volume-confirmed price direction.
    OBV above its EMA = bullish volume flow, below = bearish.
    Normalized by rolling std."""
    obv = pd.Series(talib.OBV(close_s.values, volume_s.values), index=close_s.index)
    obv_ema = pd.Series(talib.EMA(obv.values, timeperiod=ema_len), index=close_s.index)
    
    diff = obv - obv_ema
    diff_std = diff.rolling(100, min_periods=20).std().replace(0, np.nan)
    z = diff / diff_std
    return z.clip(-2, 2) / 2.0  # Map [-2, 2] -> [-1, 1]


def forecast_donchian(high_s, low_s, close_s, length=DONCH_LEN):
    """Donchian Channel: Price position within N-day high/low range.
    Near top of channel = bullish breakout, near bottom = bearish.
    This is a TREND signal (not mean reversion)."""
    upper = high_s.rolling(length).max()
    lower = low_s.rolling(length).min()
    
    channel_width = (upper - lower).replace(0, np.nan)
    position = (close_s - lower) / channel_width  # 0 to 1
    
    # Trend signal: top of channel = bullish, bottom = bearish
    score = (position - 0.5) * 2.0
    return score.clip(-1, 1)


print("8 continuous forecast generators defined:")
print("   1. TEMA          (trend direction)")
print("   2. MACD-V        (volatility-normalized momentum)")
print("   3. RSI           (mean-reversion timing)")
print("   4. ADX           (trend strength + direction)")
print("   5. Stochastic    (overbought/oversold oscillator)")
print("   6. Bollinger %B  (price vs volatility bands)")
print("   7. OBV Trend     (volume confirmation)")
print("   8. Donchian      (channel breakout)")

## 6. Generate All Forecasts & Build the Composite Score

Equal-weight average of all 8 forecasts. No optimization on weights — Carver shows equal weighting beats "optimal" weights out-of-sample almost every time.

In [ ]:
# Generate all 8 forecasts on full data
forecasts = pd.DataFrame({
    'TEMA':       forecast_tema(close),
    'MACD_V':     forecast_macdv(close, atr),
    'RSI':        forecast_rsi(close),
    'ADX':        forecast_adx(high, low, close),
    'Stochastic': forecast_stochastic(high, low, close),
    'Bollinger':  forecast_bollinger(close),
    'OBV':        forecast_obv(close, volume),
    'Donchian':   forecast_donchian(high, low, close),
}, index=close.index)

# Drop warmup NaNs
warmup_nans = forecasts.isna().any(axis=1).sum()
forecasts = forecasts.dropna()
print(f"Warmup rows dropped: {warmup_nans}")
print(f"Forecast matrix: {forecasts.shape[0]} rows x {forecasts.shape[1]} signals\n")

# Equal-weight composite score
composite = forecasts.mean(axis=1)

# Stats on each forecast
print(f"{'Signal':<12} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8} {'% Bullish':>10}")
print("-" * 58)
for col in forecasts.columns:
    s = forecasts[col]
    pct_bull = (s > 0).mean()
    print(f"{col:<12} {s.mean():>8.3f} {s.std():>8.3f} {s.min():>8.3f} {s.max():>8.3f} {pct_bull:>10.1%}")
print("-" * 58)
print(f"{'COMPOSITE':<12} {composite.mean():>8.3f} {composite.std():>8.3f} {composite.min():>8.3f} {composite.max():>8.3f} {(composite > 0).mean():>10.1%}")

## 7. Signal Correlation Matrix — The Diversification Check

Low correlation between signals = genuine diversification benefit. If all signals are >0.7 correlated, the ensemble is just one signal with extra steps.

In [ ]:
# Correlation matrix
corr = forecasts.corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)

# Annotate
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center',
                color='white' if abs(corr.iloc[i,j]) > 0.5 else 'black', fontsize=10)

plt.colorbar(im, ax=ax, label='Correlation')
ax.set_title('Forecast Signal Correlation Matrix', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# Summary stats
triu_idx = np.triu_indices(len(corr), k=1)
pairwise = corr.values[triu_idx]
avg_corr = pairwise.mean()
max_corr = pairwise.max()
min_corr = pairwise.min()

print(f"\nPairwise correlation stats:")
print(f"   Average: {avg_corr:.3f}")
print(f"   Max:     {max_corr:.3f}")
print(f"   Min:     {min_corr:.3f}")

# Carver diversification multiplier
n_signals = len(corr)
if avg_corr >= 0:
    div_mult = 1 / np.sqrt(1/n_signals + (1 - 1/n_signals) * avg_corr)
    print(f"   Carver diversification multiplier: {min(div_mult, 2.5):.2f}")
    if avg_corr < 0.3:
        print("   LOW correlation — strong diversification benefit")
    elif avg_corr < 0.5:
        print("   MODERATE correlation — decent diversification")
    else:
        print("   HIGH correlation — limited ensemble benefit")

## 8. Train / OOS Split

In [ ]:
# Use the forecast index (NaN-trimmed) for splitting
valid_idx = forecasts.index
split_pos = int(len(valid_idx) * TRAIN_RATIO)
split_date = valid_idx[split_pos]

train_idx = valid_idx[:split_pos]
test_idx  = valid_idx[split_pos:]

# Aligned data slices
train_close = close.loc[train_idx]
test_close  = close.loc[test_idx]
train_composite = composite.loc[train_idx]
test_composite  = composite.loc[test_idx]
train_regime = regime_bull.loc[train_idx]
test_regime  = regime_bull.loc[test_idx]
train_forecasts = forecasts.loc[train_idx]
test_forecasts  = forecasts.loc[test_idx]

print(f"Split at {split_date.date()} (index {split_pos})")
print(f"   TRAIN: {len(train_idx)} bars | {train_idx[0].date()} -> {train_idx[-1].date()}")
print(f"   OOS:   {len(test_idx)} bars  | {test_idx[0].date()} -> {test_idx[-1].date()}")
print(f"   OOS is SEALED until Section 11.")

## 9. Threshold Sweep on Training Data

The ONLY thing we optimize: the entry threshold for the composite score. This is ONE parameter (not hundreds like a grid search). We test a handful of sensible values.

In [ ]:
def run_composite_backtest(close_s, composite_s, regime_s, entry_thresh, exit_thresh=EXIT_THRESHOLD,
                           shift=SIGNAL_SHIFT, use_regime=True):
    """Run a backtest using the composite forecast score.
    Entry: composite > entry_thresh (and regime filter)
    Exit:  composite < exit_thresh
    Returns vbt.Portfolio or None."""
    entries = composite_s > entry_thresh
    exits = composite_s < exit_thresh
    
    if use_regime:
        entries = entries & regime_s
    
    # Anti-lookahead shift
    entries = entries.shift(shift).fillna(False).astype(bool)
    exits = exits.shift(shift).fillna(False).astype(bool)
    
    if entries.sum() < MIN_TRADES:
        return None
    
    pf = vbt.Portfolio.from_signals(close_s, entries=entries, exits=exits,
                                     init_cash=INIT_CASH, fees=FEES, freq='D')
    return pf


# Sweep entry thresholds on training data
print("Threshold sweep on TRAINING data:\n")
print(f"{'Threshold':>10} {'Sharpe':>8} {'Sortino':>8} {'Return':>10} {'MaxDD':>8} {'Trades':>8} {'WinRate':>8}")
print("-" * 70)

threshold_results = []
for thresh in ENTRY_THRESHOLDS:
    pf = run_composite_backtest(train_close, train_composite, train_regime, thresh)
    if pf is not None:
        sr = float(pf.sharpe_ratio())
        so = float(pf.sortino_ratio())
        ret = float(pf.total_return())
        dd = float(pf.max_drawdown())
        trades = int(pf.trades.count())
        wr = float(pf.trades.win_rate()) if trades > 0 else 0
        threshold_results.append({
            'threshold': thresh, 'sharpe': sr, 'sortino': so,
            'return': ret, 'max_dd': dd, 'trades': trades, 'win_rate': wr, 'pf': pf
        })
        print(f"{thresh:>10.2f} {sr:>8.4f} {so:>8.4f} {ret:>10.2%} {dd:>8.2%} {trades:>8} {wr:>8.1%}")
    else:
        print(f"{thresh:>10.2f} {'---':>8} {'---':>8} {'---':>10} {'---':>8} {'<10':>8} {'---':>8}")

# Pick best threshold by Sharpe
if threshold_results:
    best_row = max(threshold_results, key=lambda x: x['sharpe'])
    BEST_THRESH = best_row['threshold']
    pf_is = best_row['pf']
    print(f"\nBEST THRESHOLD: {BEST_THRESH:.2f} (Sharpe={best_row['sharpe']:.4f})")
    print(f"   Note: We tested {len(ENTRY_THRESHOLDS)} thresholds — this is 1 degree of freedom, not hundreds.")
else:
    print("\nNo valid threshold found — composite signal may be too weak for this asset/period.")

## 10. In-Sample Results & Equity Curve

In [ ]:
print(f"In-Sample Weak Signals Ensemble — threshold={BEST_THRESH:.2f}")
print(f"{'='*65}")
print(f"   Sharpe:        {pf_is.sharpe_ratio():.4f}")
print(f"   Sortino:       {pf_is.sortino_ratio():.4f}")
print(f"   Total Return:  {pf_is.total_return():.2%}")
print(f"   Max Drawdown:  {pf_is.max_drawdown():.2%}")
print(f"   Trades:        {pf_is.trades.count()}")
wr = pf_is.trades.win_rate() if pf_is.trades.count() > 0 else 0
print(f"   Win Rate:      {wr:.2%}")
pf_factor = pf_is.trades.profit_factor() if pf_is.trades.count() > 0 else 0
print(f"   Profit Factor: {pf_factor:.2f}")

# Plot
fig, axes = plt.subplots(3, 1, figsize=(15, 10), gridspec_kw={'height_ratios': [3, 1, 1]})
fig.suptitle(f'In-Sample Weak Signals Ensemble | {TICKER} | Threshold={BEST_THRESH}', fontsize=13, fontweight='bold')

pf_is.value().plot(ax=axes[0], color='navy', linewidth=1.5)
axes[0].set_ylabel('Portfolio Value ($)')
axes[0].set_title(f'Sharpe: {pf_is.sharpe_ratio():.3f} | Return: {pf_is.total_return():.1%} | MaxDD: {pf_is.max_drawdown():.1%} | Trades: {pf_is.trades.count()}')
axes[0].grid(alpha=0.3)

pf_is.drawdown().plot(ax=axes[1], color='crimson', linewidth=1)
axes[1].fill_between(pf_is.drawdown().index, pf_is.drawdown().values, alpha=0.3, color='crimson')
axes[1].set_ylabel('Drawdown')
axes[1].grid(alpha=0.3)

# Composite score over time
train_composite.plot(ax=axes[2], color='gray', linewidth=0.8, alpha=0.7)
axes[2].axhline(BEST_THRESH, color='green', linestyle='--', label=f'Entry ({BEST_THRESH})')
axes[2].axhline(EXIT_THRESHOLD, color='red', linestyle='--', label=f'Exit ({EXIT_THRESHOLD})')
axes[2].set_ylabel('Composite Score')
axes[2].legend(loc='upper right')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Out-of-Sample Validation — The Moment of Truth

In [ ]:
# Run on OOS with the threshold selected on training data
pf_oos = run_composite_backtest(test_close, test_composite, test_regime, BEST_THRESH)

# Also run Buy & Hold on OOS for benchmark
pf_bh = vbt.Portfolio.from_holding(test_close, init_cash=INIT_CASH, fees=FEES, freq='D')

is_sr  = float(pf_is.sharpe_ratio())
oos_sr = float(pf_oos.sharpe_ratio()) if pf_oos else float('nan')
bh_sr  = float(pf_bh.sharpe_ratio())
degrad = (1 - oos_sr/is_sr)*100 if (is_sr != 0 and not np.isnan(oos_sr)) else float('nan')

# Comparison table
print(f"IS vs OOS vs Buy & Hold — Weak Signals Ensemble (threshold={BEST_THRESH})")
print("=" * 85)
print(f"{'Metric':<20} {'In-Sample':>15} {'Out-of-Sample':>15} {'Buy & Hold OOS':>15}")
print("-" * 85)

def row(label, is_val, oos_val, bh_val):
    print(f"{label:<20} {is_val:>15} {oos_val:>15} {bh_val:>15}")

row('Sharpe', f"{is_sr:.4f}", f"{oos_sr:.4f}" if not np.isnan(oos_sr) else "N/A", f"{bh_sr:.4f}")

if pf_oos:
    row('Sortino', f"{pf_is.sortino_ratio():.4f}", f"{pf_oos.sortino_ratio():.4f}", f"{pf_bh.sortino_ratio():.4f}")
    row('Total Return', f"{pf_is.total_return():.2%}", f"{pf_oos.total_return():.2%}", f"{pf_bh.total_return():.2%}")
    row('Max Drawdown', f"{pf_is.max_drawdown():.2%}", f"{pf_oos.max_drawdown():.2%}", f"{pf_bh.max_drawdown():.2%}")
    row('Trades', str(pf_is.trades.count()), str(pf_oos.trades.count()), "1")
    oos_wr = pf_oos.trades.win_rate() if pf_oos.trades.count() > 0 else 0
    is_wr = pf_is.trades.win_rate() if pf_is.trades.count() > 0 else 0
    row('Win Rate', f"{is_wr:.1%}", f"{oos_wr:.1%}", "N/A")
    oos_pf_val = pf_oos.trades.profit_factor() if pf_oos.trades.count() > 0 else 0
    is_pf_val = pf_is.trades.profit_factor() if pf_is.trades.count() > 0 else 0
    row('Profit Factor', f"{is_pf_val:.2f}", f"{oos_pf_val:.2f}", "N/A")

print("-" * 85)
if not np.isnan(degrad):
    tag = 'Healthy' if abs(degrad) < 30 else ('Moderate' if abs(degrad) < 50 else 'SEVERE OVERFIT')
    print(f"   Sharpe Degradation IS->OOS: {degrad:.1f}% [{tag}]")
if pf_oos and oos_sr > bh_sr:
    print(f"   Ensemble BEATS Buy & Hold on OOS (Sharpe {oos_sr:.4f} vs {bh_sr:.4f})")
elif pf_oos:
    print(f"   Ensemble LOSES to Buy & Hold on OOS (Sharpe {oos_sr:.4f} vs {bh_sr:.4f})")

# Plot IS vs OOS
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle(f'Weak Signals Ensemble — IS vs OOS | {TICKER}', fontsize=13, fontweight='bold')

pf_is.value().plot(ax=axes[0], color='navy', linewidth=1.5)
axes[0].set_title(f'In-Sample | Sharpe: {is_sr:.3f}')
axes[0].set_ylabel('Portfolio Value ($)'); axes[0].grid(alpha=0.3)

if pf_oos:
    pf_oos.value().plot(ax=axes[1], color='darkgreen', linewidth=1.5, label='Ensemble')
    pf_bh.value().plot(ax=axes[1], color='gray', linewidth=1, alpha=0.7, linestyle='--', label='Buy & Hold')
    axes[1].set_title(f'Out-of-Sample | Sharpe: {oos_sr:.3f} vs B&H: {bh_sr:.3f}')
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, 'OOS: Insufficient signals', ha='center', va='center', fontsize=14)
    axes[1].set_title('Out-of-Sample | N/A')
axes[1].set_ylabel('Portfolio Value ($)'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Leave-One-Out — Which Signals Actually Contribute?

Drop each signal one at a time and re-run. If removing a signal HURTS performance, it was contributing. If removing it HELPS, it was adding noise.

In [ ]:
# Leave-one-out analysis on OOS data
print("Leave-One-Out Analysis (OOS data):\n")
print(f"{'Dropped Signal':<15} {'OOS Sharpe':>10} {'Delta':>8} {'Verdict':>12}")
print("-" * 50)

baseline_sr = oos_sr if not np.isnan(oos_sr) else 0
loo_results = []

for col in forecasts.columns:
    # Recompute composite without this signal
    remaining = [c for c in forecasts.columns if c != col]
    loo_composite = forecasts[remaining].loc[test_idx].mean(axis=1)
    
    pf_loo = run_composite_backtest(test_close, loo_composite, test_regime, BEST_THRESH)
    if pf_loo is not None:
        loo_sr = float(pf_loo.sharpe_ratio())
        delta = loo_sr - baseline_sr
        verdict = "NOISE" if delta > 0.05 else ("HELPS" if delta < -0.05 else "NEUTRAL")
        loo_results.append({'signal': col, 'oos_sharpe': loo_sr, 'delta': delta, 'verdict': verdict})
        print(f"{col:<15} {loo_sr:>10.4f} {delta:>+8.4f} {verdict:>12}")
    else:
        print(f"{col:<15} {'N/A':>10} {'N/A':>8} {'NO TRADES':>12}")

print(f"\n   Baseline (all 8 signals): OOS Sharpe = {baseline_sr:.4f}")
contributors = [r for r in loo_results if r['verdict'] == 'HELPS']
noise = [r for r in loo_results if r['verdict'] == 'NOISE']
if contributors:
    print(f"   Contributing signals: {', '.join(r['signal'] for r in contributors)}")
if noise:
    print(f"   Noise signals:        {', '.join(r['signal'] for r in noise)}")
    print(f"   Consider removing noise signals for a cleaner ensemble.")

## 13. FTMO Monte Carlo Simulation

Simulate 10,000 random 30-day trading windows drawn from OOS daily returns. Check what % of paths would pass the FTMO challenge (hit 10% profit before hitting 5% daily loss or 10% total drawdown).

In [ ]:
if pf_oos is not None and pf_oos.trades.count() > 0:
    # Get daily returns from OOS ensemble strategy
    oos_daily_returns = pf_oos.daily_returns().dropna().values
    
    n_paths = MONTE_CARLO_PATHS
    n_days = FTMO_CHALLENGE_DAYS
    account = FTMO_ACCOUNT
    target = account * FTMO_PROFIT_TARGET
    max_daily = account * FTMO_MAX_DAILY_DD
    max_total = account * FTMO_MAX_TOTAL_DD
    
    pass_count = 0
    fail_daily_count = 0
    fail_total_count = 0
    fail_time_count = 0
    final_pnls = []
    
    np.random.seed(42)
    for _ in range(n_paths):
        # Sample n_days returns with replacement
        sampled = np.random.choice(oos_daily_returns, size=n_days, replace=True)
        daily_pnl = account * sampled
        cumulative_pnl = np.cumsum(daily_pnl)
        equity = account + cumulative_pnl
        
        # Check FTMO rules
        hit_target = np.any(cumulative_pnl >= target)
        worst_daily = np.min(daily_pnl)
        max_dd_path = np.max(np.maximum.accumulate(equity) - equity)
        
        if worst_daily <= -max_daily:
            fail_daily_count += 1
        elif max_dd_path >= max_total:
            fail_total_count += 1
        elif hit_target:
            pass_count += 1
        else:
            fail_time_count += 1
        
        final_pnls.append(cumulative_pnl[-1])
    
    pass_rate = pass_count / n_paths
    final_pnls = np.array(final_pnls)
    
    print(f"FTMO Monte Carlo — {n_paths:,} paths x {n_days} days")
    print(f"{'='*60}")
    print(f"   PASS RATE:         {pass_rate:.1%} ({pass_count:,} / {n_paths:,})")
    print(f"   Fail (daily DD):   {fail_daily_count/n_paths:.1%}")
    print(f"   Fail (total DD):   {fail_total_count/n_paths:.1%}")
    print(f"   Fail (time):       {fail_time_count/n_paths:.1%}")
    print(f"\n   Final P&L stats:")
    print(f"   Mean:   ${final_pnls.mean():>10,.0f}")
    print(f"   Median: ${np.median(final_pnls):>10,.0f}")
    print(f"   Std:    ${final_pnls.std():>10,.0f}")
    print(f"   5th %:  ${np.percentile(final_pnls, 5):>10,.0f}")
    print(f"   95th %: ${np.percentile(final_pnls, 95):>10,.0f}")
    
    # Plot distribution
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(f'FTMO Monte Carlo — {TICKER} Weak Signals Ensemble', fontsize=13, fontweight='bold')
    
    axes[0].hist(final_pnls, bins=80, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0].axvline(target, color='green', linestyle='--', linewidth=2, label=f'Target (+${target:,.0f})')
    axes[0].axvline(-max_total * account / FTMO_ACCOUNT, color='red', linestyle='--', linewidth=2, label=f'Max Loss (-${max_total:,.0f})')
    axes[0].axvline(0, color='black', linewidth=1)
    axes[0].set_xlabel('Final P&L ($)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title(f'P&L Distribution | Pass Rate: {pass_rate:.1%}')
    axes[0].legend()
    
    # Sample equity paths
    np.random.seed(0)
    for i in range(min(100, n_paths)):
        sampled = np.random.choice(oos_daily_returns, size=n_days, replace=True)
        equity = account + np.cumsum(account * sampled)
        color = 'green' if equity[-1] > account + target else ('red' if equity[-1] < account - max_total else 'gray')
        axes[1].plot(equity, color=color, alpha=0.1, linewidth=0.5)
    axes[1].axhline(account + target, color='green', linestyle='--', label='Target')
    axes[1].axhline(account - max_total, color='red', linestyle='--', label='Max Loss')
    axes[1].set_xlabel('Trading Day')
    axes[1].set_ylabel('Equity ($)')
    axes[1].set_title('Sample Equity Paths (100)')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("Cannot run FTMO Monte Carlo — no OOS trades.")

## 14. Composite Score Visualization — What the Model "Sees"

Plot the composite score over time overlaid on price. This shows when the ensemble is bullish vs bearish, and how that aligns with actual price moves.

In [ ]:
# Full composite score + price visualization
fig, axes = plt.subplots(3, 1, figsize=(16, 12), gridspec_kw={'height_ratios': [2, 1, 1]})
fig.suptitle(f'Weak Signals Ensemble — Full History | {TICKER}', fontsize=14, fontweight='bold')

# Price + regime
valid_close = close.loc[forecasts.index]
axes[0].plot(valid_close.index, valid_close.values, color='black', linewidth=1, label='Close')
sma_valid = sma_200.loc[forecasts.index]
axes[0].plot(sma_valid.index, sma_valid.values, color='orange', linewidth=1, alpha=0.7, label='SMA-200')
# Shade bull/bear
bull_mask = regime_bull.loc[forecasts.index]
axes[0].fill_between(valid_close.index, valid_close.min(), valid_close.max(),
                      where=bull_mask, alpha=0.05, color='green')
axes[0].axvline(split_date, color='purple', linestyle='--', linewidth=2, label='Train/OOS Split')
axes[0].set_ylabel('Price ($)')
axes[0].legend(loc='upper left')
axes[0].grid(alpha=0.3)

# Composite score
axes[1].plot(composite.index, composite.values, color='steelblue', linewidth=0.8)
axes[1].fill_between(composite.index, 0, composite.values,
                      where=composite > BEST_THRESH, alpha=0.3, color='green', label='Long zone')
axes[1].fill_between(composite.index, 0, composite.values,
                      where=composite < EXIT_THRESHOLD, alpha=0.3, color='red', label='Exit zone')
axes[1].axhline(BEST_THRESH, color='green', linestyle='--', alpha=0.7)
axes[1].axhline(EXIT_THRESHOLD, color='red', linestyle='--', alpha=0.7)
axes[1].axvline(split_date, color='purple', linestyle='--', linewidth=2)
axes[1].set_ylabel('Composite Score')
axes[1].legend(loc='upper right')
axes[1].grid(alpha=0.3)

# Individual signal heatmap
signal_data = forecasts.T
im = axes[2].imshow(signal_data.values, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1,
                     extent=[0, len(signal_data.columns), len(signal_data.index), 0])
axes[2].set_yticks(np.arange(len(forecasts.columns)) + 0.5)
axes[2].set_yticklabels(forecasts.columns, fontsize=9)
# Mark split point
split_x = split_pos
axes[2].axvline(split_x, color='purple', linestyle='--', linewidth=2)
axes[2].set_xlabel('Time (bar index)')
axes[2].set_ylabel('Signal')
plt.colorbar(im, ax=axes[2], label='Forecast', shrink=0.8)

plt.tight_layout()
plt.show()

## 15. Final Audit & Honest Assessment

In [ ]:
print("=" * 80)
print("FINAL AUDIT — WEAK SIGNALS ENSEMBLE")
print("=" * 80)

print("\nDEGREES OF FREEDOM AUDIT:")
print(f"   Indicator parameters optimized: 0 (all textbook defaults)")
print(f"   Ensemble thresholds tested:     {len(ENTRY_THRESHOLDS)} (selected 1)")
print(f"   Total degrees of freedom:       1")
print(f"   Compare: Original notebook had  ~{9*7*8 + 4*5*4 + 7*3*3} indicator combos + 3 voting modes")

print("\nOVERFIT RISK ASSESSMENT:")
if not np.isnan(degrad):
    if abs(degrad) < 20:
        print(f"   Sharpe degradation: {degrad:.1f}% — LOW RISK (excellent generalization)")
    elif abs(degrad) < 40:
        print(f"   Sharpe degradation: {degrad:.1f}% — MODERATE RISK")
    else:
        print(f"   Sharpe degradation: {degrad:.1f}% — HIGH RISK (possible overfit even with 1 DOF)")

print("\nSIGNAL DIVERSITY:")
print(f"   Average pairwise correlation: {avg_corr:.3f}")
if avg_corr < 0.3:
    print(f"   STRONG diversification — signals are genuinely different")
elif avg_corr < 0.5:
    print(f"   DECENT diversification — some redundancy but acceptable")
else:
    print(f"   WEAK diversification — signals too correlated, ensemble adds limited value")

print("\nFTMO VIABILITY:")
if 'pass_rate' in dir():
    if pass_rate > 0.30:
        print(f"   Pass rate: {pass_rate:.1%} — VIABLE for FTMO challenge")
    elif pass_rate > 0.15:
        print(f"   Pass rate: {pass_rate:.1%} — MARGINAL (need more edge or better risk management)")
    else:
        print(f"   Pass rate: {pass_rate:.1%} — NOT VIABLE for FTMO as-is")

print("\nKNOWN LIMITATIONS:")
print("   1. Single asset — test on ETH, SOL, QQQ, DAX before any real deployment")
print("   2. Binary position sizing (100% in/out) — no vol targeting")
print("   3. No FTMO daily drawdown limit enforced in backtest")
print("   4. RSI/Stochastic/Bollinger are mean-reversion in a trend-following regime filter")
print("      This is a FEATURE (buy dips in uptrends) but may contradict in ranging markets")
print("   5. OBV volume data may be unreliable for crypto (exchange-specific)")

print("\nNEXT STEPS:")
print("   1. Run on 3+ assets to check robustness")
print("   2. Add volatility-targeted position sizing (Carver Ch. 10)")
print("   3. If leave-one-out shows noise signals, test a trimmed ensemble")
print("   4. Walk-forward validation (roll the train/test window)")
print("   5. Paper trade for 3+ months before any capital")
print("=" * 80)